# Wall-time gate — does JAX first-call compile dominate real training?

`bench_gradient.py` reports per-step medians *after* warm-up, so the JAX compile cost is hidden. This notebook runs a full gradient-descent loop for **1, 10, 100, 500** iterations and reports total wall seconds. That's what a user actually experiences when training.

**Run on GPU.** CPU/TPU numbers don't change the decision for plan/15 (Track B).

Single result cell at the end — paste its output back.

**What we're looking for:**
- At iters=1 JAX should be ~2–3 s (compile dominates).
- At iters=100 torch/jax ratio should match `bench_gradient.py` (compile amortised).
- If iters=500 still shows compile dominating the per-iter column, we have a problem.

In [ ]:
import os, platform, subprocess, sys

BRANCH = 'plan-14-jax-port'
REPO   = 'https://github.com/keunjunpark/qiskit-trev.git'

def _detect():
    try:
        import jax as _j
        if any('tpu' in type(d).__name__.lower() for d in _j.devices()):
            return 'tpu'
    except Exception:
        pass
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                             capture_output=True, text=True, check=True).stdout.strip()
        if out:
            return f'gpu:{out.splitlines()[0].strip()}'
    except Exception:
        pass
    return 'cpu'

ACCEL = _detect()
print('Detected accelerator:', ACCEL)
print('Python   :', sys.version.split()[0])

In [ ]:
def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

if ACCEL == 'tpu':
    _pip('jax[tpu]', '-f', 'https://storage.googleapis.com/jax-releases/libtpu_releases.html')
elif ACCEL.startswith('gpu'):
    _pip('jax[cuda12]')
else:
    _pip('jax')

import jax
print('jax', jax.__version__, '|', 'devices:', jax.devices())

In [ ]:
import shutil
REPO_DIR = '/tmp/qiskit-trev'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO, REPO_DIR], check=True)
_pip('-e', REPO_DIR)

# Prepend src/ so an already-cached sys.path picks up the editable install.
SRC = os.path.join(REPO_DIR, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
for mod in list(sys.modules):
    if mod.startswith('qiskit_trev'):
        del sys.modules[mod]

import qiskit_trev
print('qiskit_trev from:', qiskit_trev.__file__)
print('HEAD:', subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--oneline'],
                               capture_output=True, text=True).stdout.strip())

In [ ]:
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('torch gpu :', torch.cuda.get_device_name(0))

In [ ]:
import contextlib, io, runpy, time

buf = io.StringIO()
t0 = time.perf_counter()
err = None
try:
    with contextlib.redirect_stdout(buf):
        runpy.run_path(f'{REPO_DIR}/bench/bench_training_wall_time.py', run_name='__main__')
except Exception as e:
    err = repr(e)
print(buf.getvalue())
if err:
    print('--- ERROR ---')
    print(err)
print(f'\n(total cell wall time: {time.perf_counter() - t0:.1f} s)')

In [ ]:
import json
slug = ACCEL.replace(':', '_').replace(' ', '_')
out_path = f'/tmp/qiskit_trev_wall_time_{slug}.json'
with open(out_path, 'w') as f:
    json.dump({
        'accelerator': ACCEL,
        'jax_version': jax.__version__,
        'jax_devices': [str(d) for d in jax.devices()],
        'torch_version': torch.__version__,
        'torch_cuda': bool(torch.cuda.is_available()),
        'torch_gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        'platform': platform.platform(),
        'branch': BRANCH,
        'stdout': buf.getvalue(),
    }, f, indent=2)
print('wrote', out_path)